In [1]:
import numpy as np
import astropy 
from astropy.io import fits, ascii 
from astropy.table import Table, Column
import subprocess
import os

In [2]:
# read in files
path = '../data/'
hdu = fits.open(path + "imbh_sample.fits")
imbh = hdu[1].data
hdu = fits.open(path+'spAll-v6_1_3.fits')
dr19_download = hdu[1].data

In [17]:
sp = ascii.read('./sdss_softening_param.txt')
zcutoff=0.7
def sdss_flux_mag(f, band): 
    idx = np.where(sp['filter']=='i')[0][0]
    b = sp['b'][idx]
    return -2.5/np.log10(10) * (np.asinh((f/1E9)/(2*b)) + np.log10(b))
def lrg_search(source):
    sdss_i = sdss_flux_mag(source['SDSS_CALIBFLUX_i'], 'i')
    sdss_z = sdss_flux_mag(source['SDSS_CALIBFLUX_z'], 'z')
    sdss_r = sdss_flux_mag(source['SDSS_CALIBFLUX_r'], 'r')
    # definitions from SDSS DR17 
    lrg_izw = (sdss_i - sdss_z > 0.7) & (sdss_i - source['WISE_w1mpro'] > (2.143)*(sdss_i -sdss_z) - 0.2) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg_riw = (sdss_r - sdss_i > 0.98) & (sdss_r - source['WISE_w1mpro'] > 2*(sdss_r - sdss_i)) & (sdss_i - sdss_z > 0.625) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg = (lrg_izw) | (lrg_riw)
    return(lrg)

In [20]:
for source in imbh: 
    if(np.isnan(source['DESI_TARGETID'])): 
        if(source['dr'] == 17): 
            survey = 'dr17'
        elif(source['dr'] == 19): 
            survey = 'dr19'
        else: 
            survey = None
    else: 
        survey = 'desi'
    highz = bool(source['combo_Z'] >= zcutoff)
    if(highz): 
        lrg = lrg_search(source)
        if(lrg): 
            fit_continuum = True 
        else: 
            fit_continuum = False 
    else: 
        fit_continuum = True 
    ## find file 
    if(survey == 'dr17'): 
        file = f"spec-{int(source['plate']):04d}-{int(source['mjd']):05d}-{int(source['fiberid']):04d}.fits"
    if(survey == 'dr19'): 
        specobjid = source['SDSS_SPECOBJID'].split("'")[1].strip()
        m = (specobjid == dr19_download['SPECOBJID'])
        idx = np.where(m == True)
        file = dr19_download['SPEC_FILE'][idx][0]
    if(survey == 'dr17' or survey == 'dr19'): 
        path = '/Users/f007znp/Research/processed/SDSS_BOSS/'
    if(fit_continuum): 
        if(survey == 'dr19'): 
            with open("fit_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['field']}\n")
        elif(survey == 'dr17'): 
            with open("fit_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['plate']}\n")
    elif(~fit_continuum): 
        if(survey == 'dr19'): 
            with open("no_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['field']}\n")
        elif(survey == 'dr17'): 
            with open("no_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['plate']}\n")

/var/folders/m1/ly87n3sn4g31x85g5tlkr4b80000gp/T/ipykernel_63766/2553811192.py:37: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  elif(~fit_continuum):


Run in gdl 

In [ ]:
process_survey,'sdss',inptable='../IMBH/nburst_fittingscript/no_continuum.txt',/plot,nlosvd=3,emexcl=0,emlt1=[1],emlt2=[2],moments=4,start=[0,100,0,80,0,400,3000,-1.2],siglimits=[500,500,3000],/disable_stpop,/xsl,lammin=3700,lammax=9000,degree=2,mdegree=5,path_ssp='/Users/f007znp/Research/stellar_templates/XSL/Kroupa/',prefix='SB_',suffix='_XSL_Kroupa_PC.fits',outpath='SDSS_BOSS/scripttest/'

In [ ]:
source = imbh[2]
if(np.isnan(source['DESI_TARGETID'])): 
    if(source['dr'] == 17): 
        survey = 'dr17'
    elif(source['dr'] == 19): 
        survey = 'dr19'
    else: 
        survey = None
else: 
    survey = 'desi'

In [6]:
zcutoff=0.7
def sort_by_z(source): 
    highz = bool(source['combo_Z']) >= zcutoff
    return highz 
highz = sort_by_z(source)

if(highz): 
    lrg = lrg_search(source)
    if(lrg): 
        fit_continuum = True 
    else: 
        fit_continuum = False 
else: 
    fit_continuum = True 

## find file 
if(survey == 'dr17'): 
    file = f"spec-{source['plate']:04d}-{source['mjd']:05d}-{source['fiberid']:04d}.fits"
if(survey == 'dr19'): 
    specobjid = source['SDSS_SPECOBJID'].split("'")[1].strip()
    m = (specobjid == dr19_download['SPECOBJID'])
    idx = np.where(m == True)
    file = dr19_download['SPEC_FILE'][idx][0]
if(survey == 'dr17' or survey == 'dr19'): 
    path = '/Users/f007znp/Research/processed/SDSS_BOSS/'
if(fit_continuum): 
    if(survey == 'dr19'): 
        with open("fit_continuum.txt", 'a') as f: 
            f.write(f"{path}{file} {source['field']}\n")
    elif(survey == 'dr17'): 
        with open("fit_continuum.txt", 'a') as f: 
            f.write(f"{path}{file} {source['plate']}\n")
else: 
    if(survey == 'dr19'): 
        with open("no_continuum.txt", 'a') as f: 
            f.write(f"{path}{file} {source['field']}\n")
    elif(survey == 'dr17'): 
        with open("no_continuum.txt", 'a') as f: 
            f.write(f"{path}{file} {source['plate']}\n")

In [ ]:
gdl('cd, current=cwd')
gdl('print, cwd')

In [ ]:
zcutoff=0.7
def split_cat_by_z(catalog): 
    m = catalog['combo_Z']>= zcutoff
    highz = catalog[m]
    lowz = catalog[~m]
    return highz, lowz 

In [ ]:
dr17_highz, dr17_lowz = split_cat_by_z(dr17)
dr19_highz, dr19_lowz = split_cat_by_z(dr19)
desi_highz, desi_lowz = split_cat_by_z(desi)

In [ ]:
## for SDSS LRG definition ! 
sp = ascii.read('./sdss_softening_param.txt')
def sdss_flux_mag(f, band): 
    idx = np.where(sp['filter']=='i')[0][0]
    b = sp['b'][idx]
    return -2.5/np.log10(10) * (np.asinh((f/1E9)/(2*b)) + np.log10(b))
def lrd_search(catalog):
    sdss_i = sdss_flux_mag(catalog['SDSS_CALIBFLUX_i'], 'i')
    sdss_z = sdss_flux_mag(catalog['SDSS_CALIBFLUX_z'], 'z')
    sdss_r = sdss_flux_mag(catalog['SDSS_CALIBFLUX_r'], 'r')
    # definitions from SDSS DR17 
    lrg_izw = (sdss_i - sdss_z > 0.7) & (sdss_i - catalog['WISE_w1mpro'] > (2.143)*(sdss_i -sdss_z) - 0.2) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg_riw = (sdss_r - sdss_i > 0.98) & (sdss_r - catalog['WISE_w1mpro'] > 2*(sdss_r - sdss_i)) & (sdss_i - sdss_z > 0.625) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg = (lrg_izw) | (lrg_riw)
    return(catalog[lrg])
dr17_lrg = lrd_search(dr17_highz)
dr19_lrg = lrd_search(dr19_highz)

In [ ]:
# 